# cv-pilznet – Training auf Google Colab

Dünnes Runner-Notebook: holt Repo-Code und Datensatz, der Rest (Datenpipeline, Training, Auswertung) passiert in ganz normalen Python-Modulen unter `src/`.

**Vorher:** Laufzeittyp auf GPU stellen (Laufzeit -> Laufzeittyp ändern -> GPU/T4).


In [ ]:
!nvidia-smi

## 1. Repo-Code holen

Das GitLab-Repo ist privat. Entweder einen Personal Access Token in die URL einsetzen (`https://oauth2:<TOKEN>@gitlab.bht-berlin.de/...`), oder den `src/`-Ordner einfach über das Colab-Dateiupload-Icon hochladen und diese Zelle überspringen.


In [ ]:
GITLAB_TOKEN = ""  # Personal Access Token hier eintragen (read_repository reicht)
REPO_URL = "https://gitlab.bht-berlin.de/s84985/cv-pilznet.git"
clone_url = REPO_URL.replace("https://", f"https://oauth2:{GITLAB_TOKEN}@") if GITLAB_TOKEN else REPO_URL

!git clone {clone_url}
%cd cv-pilznet

## 2. Abhängigkeiten

Torch/torchvision/sklearn/pandas/matplotlib sind in Colab schon drauf, nur `albumentations` und die `kaggle`-CLI fehlen.


In [ ]:
!pip install -q albumentations kaggle

## 3. Datensatz laden

`kaggle.json` hochladen (API-Token, siehe README für die Einrichtung), dann Datensatz runterladen und entpacken.


In [ ]:
from google.colab import files

uploaded = files.upload()  # kaggle.json auswählen
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d zlatan599/mushroom1 -p data/raw --unzip

## 4. Labels aufbereiten


In [ ]:
!python src/build_labels.py

## 5. Training

Beide Architekturen für den Vergleich trainieren. Batch-Size ist schon auf 96 hochgesetzt und Epochs auf 8 runter (Early Stopping bricht eh meistens vorher ab) – damit geht's in Colab spürbar schneller. Parameter bei Bedarf anpassen.


In [ ]:
!python src/train.py --arch mobilenet_v2 --epochs 8 --freeze-epochs 3 --batch-size 96


In [ ]:
!python src/train.py --arch resnet50 --epochs 8 --freeze-epochs 3 --batch-size 96


## 6. Ergebnisse runterladen

Checkpoints (`models/*.pt`) und Trainingshistorien (`results/*_history.json`) runterladen, um `src/evaluate.py` lokal auf der CPU laufen zu lassen – für reine Inferenz reicht das locker, spart Colab-GPU-Zeit.


In [ ]:
!zip -r run_output.zip models results
from google.colab import files
files.download("run_output.zip")